# MCM LTER Hyporheic Sensor Data Processing

This script was created by Jared Collins in 2025 to be used by the MCM LTER stream team as an easy way to process data from the hyporheic sensors from the stream gages, most of which were installed in the 2425 season. The sensors measure volumetric water content (VWC), conductivity, and temperature in the sediments beneath the stream.

## Importing Packages + Setting Directory

It is very important that the computer being used for processing contains the proper packages, paths, and names for the data files that will be used. The hyporheic data is a little bit more difficult to work with than data processed in Aquarius- unlike data from Aquarius, it is not automatically converted into a dataframe from the .dat file.

**To use this script, please copy the .dat file you would like to convert to your downloads folder.**

Note: The datafiles from the CR1000X all have the same name with no date differentiation, so this can only be done using one data file per stream at a time.

In [3]:
from datetime import date
import pandas as pd
import os
from pathlib import Path

In [4]:
home = Path.home()
downloads_path = home / "Downloads"
downloads_path = downloads_path.as_posix()
print("Downloads folder:", downloads_path)

Downloads folder: C:/Users/jared/Downloads


In [5]:
streamid_map = {
    'B3': 'lawson_b3',
    'B5': 'bohner_b5',
    'C1': 'common_c1',
    'F1': 'canada_f1',
    'F2': 'huey_f2',
    'F3': 'lostseal_f3',
    'F5': 'aiken_f5',
    'F6': 'vonguerard_f6',
    'F7': 'harnish_f7',
    'F8': 'crescent_f8',
    'F9': 'green_f9',
    'F10': 'delta_f10',
    'M1': 'adams_m1',
    'M2': 'miers_m2',
    'Onyx_LWRT': 'onyx_lwright',
    'Onyx_Vanda': 'onyx_vnda'}

start_map = {
    'B3': 'B3_Lawson',
    'B5': 'B5_Bohner',
    'C1': 'C1_Commonwealth',
    'F1': 'F1_Canada',
    'F2': 'F2_Huey',
    'F3': 'F3_Lost Seal',
    'F5': 'F5_Aiken',
    'F6': 'F6_Von Guerard',
    'F7': 'F7_Harnish',
    'F8': 'F8_Crescent',
    'F9': 'F9_Green',
    'F10': 'F10_Delta',
    'M1': 'M1_Adams',
    'M2': 'M2_Miers',
    'Onyx_LWRT': 'Onyx@LWRTt',
    'Onyx_Vanda': 'Onyx@Vanda'}

end_map = {'B3': 'B3_Lawson',
    'B5': 'B5_Bohner',
    'C1': 'C1_Commonwealth',
    'F1': 'F1_Canada',
    'F2': 'F2_Huey',
    'F3': 'F3_LostSeal',
    'F5': 'F5_Aiken',
    'F6': 'F6_VonGuerard',
    'F7': 'F7_Harnish',
    'F8': 'F8_Crescent',
    'F9': 'F9_Green',
    'F10': 'F10_Delta',
    'M1': 'M1_Adams',
    'M2': 'M2_Miers',
    'Onyx_LWRT': 'Onyx@LWRT',
    'Onyx_Vanda': 'Onyx@Vanda'}

## Specifying Data (EDIT THIS SECTION ONLY)

In order for this script to work properly, you will need to input necessary information about the files that you are hoping to publish:
- Stream ID: F1, F2, F3, F5, F6, F7, F8, F9, F10, H1, B3, B5, C1, M1, M2, Onyx_LWRT, Onyx_Vanda
- Season: Separate seasons into A and B data depending on what portion of the data you are working on; will only impact how datafile is saved. Record the season as the two years followed by the letter (for example, A data from 2022-2023 should be writted as (2223A)

**Edit the following cell with the information above. If done correctly, this is the only cell you will have to edit.**

In [6]:
streamid = 'F1'
season = '2425A'

## Importing and Downloading Data

This section will import the datafile into the python script and download a processed version of the file to your downloads folder.

**The next section will not have to be edited so long as the correct information is included above.**

In [7]:
hypo = pd.read_csv(f'{downloads_path}/{start_map[streamid]} (cr1000x)_{end_map[streamid]}.dat', skiprows = [0, 2, 3], sep = ',', low_memory = False)
streamid_lc = streamid.lower()

def clean_sensor_failures(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    bad_values = [7999, -7999, 9999, -9999, '7999', '-7999', '9999', '-9999', '-INF', 'INF','NAN','nan']
    df.replace(bad_values, np.nan, inplace=True)
    df = df.replace([np.inf, -np.inf], np.nan)
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col] = df[col].astype(float)
    return df

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/jared/Downloads/F1_Canada (cr1000x)_F1_Canada.dat'

In [29]:
subm_file = pd.DataFrame()
subm_file['dataset_code'] = [[f'strm_hydro_{streamid_lc}'] * len(dis)] * len(hypo)
subm_file['strmgageid'] = [streamid_map[streamid]] * len(hypo)
subm_file['date_time'] = hypo['TIMESTAMP']
subm_file['hzvwc_m3m3'] = hypo['hyporheic_VWC_Avg']
subm_file['hzec_dsm'] = hypo['hyporheic_EC_Avg']
subm_file['hztemp_degc'] = hypo['hyporheic_T_Avg']
subm_file = clean_sensor_failures(subm_file)
subm_file.to_csv(f'{downloads_path}/{streamid}_{season}_Hyporheic_SUBM.csv', index = False, na_rep = '')